# Model Context Protocol (MCP)

模型上下文协议（MCP）是一个 **开放协议**，标准化了应用程序向 LLMs 提供 **工具** 和 **上下文** 的方式。

LangChain Agent 可以通过 `langchain-mcp-adapters` 库使用 MCP 服务器上定义的工具。

## 安装

In [ ]:
# pip install langchain-mcp-adapters
# 或使用 uv
# uv add langchain-mcp-adapters

## 核心概念

MCP 主要提供三类资源：

- **Tools（工具）**：转换为 LangChain 工具，支持结构化内容和多模态响应
- **Resources（资源）**：作为 Blob 对象暴露，支持文本和二进制内容
- **Prompts（提示）**：加载为 LangChain 消息

## 传输方式

MCP 支持两种传输协议：

| 传输类型 | 说明 |
|---------|------|
| `stdio` | 用于本地子进程通信 |
| `streamable-http` | 用于远程服务器 |

## 连接到 MCP 服务器

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio

# 定义连接配置
connections = {
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["/path/to/math_server.py"]
    },
    "weather": {
        "transport": "http",
        "url": "http://localhost:8000/mcp"
    }
}

async def main():
    client = MultiServerMCPClient(connections)
    tools = await client.get_tools()
    return tools

# 获取工具列表
# tools = asyncio.run(main())

## 有状态会话

MultiServerMCPClient 默认是无状态的，每次调用工具都会创建新的 MCP ClientSession。

对于需要持久连接的场景，可以使用 `client.session()` 方法：

In [ ]:
# 通过 session 保持持久连接（适用于 stdio 传输）
# session = client.session("server_name")
# await session.call_tool("tool_name", arguments)

## 工具拦截器（Tool Interceptors）

工具拦截器提供了一种中间件式的控制机制，可以在 MCP 工具执行时：

- 访问运行时上下文
- 修改请求
- 实现重试逻辑
- 注入认证信息

In [ ]:
from langchain_mcp_adapters.client import ToolInterceptor
from typing import Any

class AuthInterceptor(ToolInterceptor):
    """认证拦截器示例"""
    
    async def intercept(
        self,
        tool_name: str,
        arguments: dict[str, Any],
        context: dict[str, Any]
    ) -> dict[str, Any]:
        # 在此处添加认证信息
        arguments["auth_token"] = context.get("token")
        return arguments

# 使用拦截器
# client = MultiServerMCPClient(connections, interceptors=[AuthInterceptor()])

## 创建 MCP 服务器

使用 `fastmcp` 库可以快速创建 MCP 服务器：

In [ ]:
from fastmcp import FastMCP

# 创建 MCP 服务器（stdio 传输）
mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
from fastmcp import FastMCP

# 创建 MCP 服务器（HTTP 传输）
mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location"""
    return f"The weather in {location} is sunny"

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)

## 与 Agent 集成

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os

async def setup_agent():
    # 1. 连接到 MCP 服务器
    client = MultiServerMCPClient({
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["/path/to/math_server.py"]
        }
    })
    
    # 2. 获取工具
    tools = await client.get_tools()
    
    # 3. 创建 Agent
    llm = ChatOpenAI(
        model="Qwen/Qwen2.5-7B-Instruct",
        base_url="https://api.siliconflow.cn/v1",
        api_key=os.environ.get("SILICONFLOW_API_KEY"),
        temperature=0.2,
    )
    
    agent = create_agent(llm, tools)
    return agent

# 运行 Agent
# agent = asyncio.run(setup_agent())
# result = agent.invoke({"input": "Calculate 15 + 27"})

## 总结

MCP 协议为 LangChain Agent 提供了一种标准化的方式来使用外部工具：

1. **安装** `langchain-mcp-adapters` 库
2. **配置** MultiServerMCPClient 连接到一个或多个 MCP 服务器
3. **获取** 工具并与 LangChain Agent 集成
4. **可选** 使用工具拦截器实现认证、重试等功能

## 参考资源

- [LangChain MCP 文档](https://docs.langchain.com/oss/python/langchain/mcp)
- [MCP 官方文档](https://modelcontextprotocol.io/introduction)
- [langchain-mcp-adapters GitHub](https://github.com/langchain-ai/langchain-mcp-adapters)